# Multi-AgentSystems: CrewAI & Agno

**Module 11 · Lesson 11.5**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- One Agent vs a Team: the Multi-Agent Tax
- Roles & Specialization
- Coordination Topologies
- CrewAI: Role-Based Crews
- Agno: Lightweight Teams & the Four Modes
- Handoffs & the Aggregation Problem

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## One writer, or a whole newsroom?

> 💡 **Info**
>
> A single agent can research a course, weigh the trade-offs, and write the advice - all by itself. So can one journalist write a story. But a **newsroom** - a reporter who gathers facts, an analyst who spots the angle, an editor who polishes - often produces something better, because each person is a specialist and they hand work to each other. That is the promise of **multi-agent** systems: specialized agents, each with a role and its own tools, collaborating on one task. The catch is the **bill**: a team of agents uses several times the tokens of one agent, and can make coordination mistakes - two agents choosing conflicting answers - that a single mind never would. This lesson builds the teams (from scratch, then in CrewAI and Agno) and teaches the one skill that separates good multi-agent engineers from expensive ones: knowing when a team is worth it, and when one great agent wins.

### 🎯 What you will be able to do after this lesson

- **Build** a multi-agent team from scratch (roles + topologies) and in both CrewAI and Agno.

- **Compare** the coordination topologies (sequential / route / broadcast / debate) and the two frameworks.

- **Operate** handoffs and shared state, and reconcile the aggregation problem when parallel agents conflict.

- **Evaluate** when a team is worth its token tax vs a single agent, using the Anthropic and Cognition findings.

> 📦 **Info**
>
> ✅ Before you startThis builds on **11.2** (the decision ladder - orchestrator-workers is the last single-agent rung before multi-agent; and routing) and **11.1** (reliability compounds, so more hops means more failure surface). You should know what a single agent is. This lesson is the multi-agent **patterns and two frameworks**; human-in-the-loop is Lesson 11.10 and agent evaluation is Lesson 11.11.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 📰 **Analogy**
>
> A multi-agent team is a **newsroom**. The **reporter** gathers facts (tools), the **analyst** finds the angle, the **editor** polishes the copy - each a specialist, passing work down the line. For a big investigation the newsroom wins. But for a one-line news alert, a single reporter is faster and far cheaper. **Where it breaks down:** if two reporters write the same section without talking, they contradict each other and an editor has to reconcile it - the coordination cost a single writer never pays.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“More agents is always better.”** A team costs roughly three to ten times the tokens, and with an equal budget a single agent often matches it. Default to one; add agents only when the task earns it.
> - **“Parallel agents just make it faster.”** On coupled work (like writing one feature), parallel agents act on partial context and make conflicting choices a later step must clean up. Parallelism helps only when the subtasks are genuinely independent.

> 💡 **Info**
>
> ⚠️ Anti-patternSplitting a single, coherent task across a crew “to be thorough.” You pay the token tax and invite coordination bugs for a task one well-designed agent would do better. Reach for a team only for independent parallel work, context isolation, or a critic/reviewer - not by default.

---

## 🎯 Concept 1: One Agent vs a Team: the Multi-Agent Tax

### One Agent vs a Team: the Multi-Agent Tax

A team of specialists can be better - but it costs several times the tokens. Default to one agent; escalate when the task earns it.

Start with the cost, because it governs every other decision. Each agent in a team carries its own **role prompt** in every call, and the agents pass **coordination messages** (handoffs, summaries) between them. So a team does not cost a little more than one agent - it costs **roughly three to ten times the tokens** for equivalent work (Anthropic reports even higher versus a plain chat). And here is the uncomfortable finding: when researchers hold the *thinking-token budget equal*, a single agent often matches or beats a multi-agent setup - so a lot of what looks like “coordination winning” is really just *more tokens* winning. The rule that falls out: **default to a single agent**, measure it, and add agents only when the task genuinely needs what a team provides.

> 💰 **Analogy**
>
> It is **hiring a team versus a freelancer**. A five-person crew will out-produce one freelancer on a big project - but you pay five salaries plus the meetings between them. For a small job, the freelancer delivers the same result for a fraction of the cost. You hire the team only when the work is big enough to earn all those salaries.

You add two more agents to a one-agent system for the same task. Roughly what happens to token cost?

**📝 `01_multi_agent_tax.py`** - *runnable*

In [ ]:
# THE MULTI-AGENT TAX: more agents = more tokens. A rough model of why coordination is not free.
def cost(n_agents, task_tokens=200, role_tokens=150, coord_tokens=80):
    per_agent = task_tokens + role_tokens               # each agent re-reads the task + its own role prompt
    coordination = coord_tokens * max(0, n_agents - 1)  # handoff / summary messages between agents
    return n_agents * per_agent + coordination

for n in (1, 3, 5):
    c = cost(n)
    print(f"  {n} agent(s): ~{c:4d} tokens  ({c / cost(1):.1f}x a single agent)")
print("rule: hold the token budget equal and one agent often matches a team -")
print("      pay for extra agents only when the task actually needs them.")

# Output:
#   1 agent(s): ~ 350 tokens  (1.0x a single agent)
#   3 agent(s): ~1210 tokens  (3.5x a single agent)
#   5 agent(s): ~2070 tokens  (5.9x a single agent)
# rule: hold the token budget equal and one agent often matches a team -
#       pay for extra agents only when the task actually needs them.

- Each agent re-reads the task and carries its own role prompt - that is the per-agent cost.
- Coordination messages between agents add on top, growing with the team size.
- A 3-agent team costs about 3.5x a single agent here; a 5-agent team about 5.9x - the tax is real.
- This is a rough model, but the direction is the point: hold the budget equal and one agent often matches a team.

#### 💡 What just happened

⚡ What just happened? A team of agents costs roughly three to ten times the tokens of one agent, and an equal-budget single agent often matches it. Tradeoff / rule: default to a single agent and escalate only when the task earns the tax. The next steps show what a team buys you when it IS worth it.

- Slide the agent count and watch tokens and the multiplier climb
- A value threshold shows when a team is actually worth it

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Roles & Specialization

### Roles & Specialization

An agent is role + goal + backstory (+ tools). The backstory is a system prompt that shapes HOW it thinks.

What makes a “specialist” agent? Three things beyond its tools: a **role** (who it is), a **goal** (what it is trying to do), and a **backstory**. The backstory is the surprising one: it is not flavour text - it is a **system prompt injected into every call** that shapes how the agent reasons. Tell one agent it is “a career counselor with ten years in tech hiring” and it weighs job outcomes; tell another it is “a skeptic who finds the risks” and it pushes back. Below, two specialists answer the *same* question and disagree - purely because their backstories point them at different things. That is the whole idea of a team: put the right lens on each part of the work. (And remember Step 1: every backstory is tokens you pay on every call.)

> 💼 **Analogy**
>
> The backstory is a **job description**. Hire someone as ‘a fact-checker who trusts nothing without a source’ and they behave differently from someone hired as ‘a big-picture strategist.’ You are not decorating the agent - you are hiring it into a role, and the role changes what it does.

The Optimist and the Skeptic get the exact same question. Will they give the same answer?

**📝 `02_roles.py`** - *runnable*

In [ ]:
# A SPECIALIST = role + goal + BACKSTORY (+ tools). The backstory is a system prompt that shapes
# HOW the agent reasons - so the same question gets different answers from different specialists.
class Specialist:
    def __init__(self, name, backstory, lens):
        self.name, self.backstory, self.lens = name, backstory, lens
    def answer(self, question):            # SCRIPTED think step - a real agent calls an LLM here
        return f"[{self.name}] {self.lens(question)}"

optimist = Specialist("Optimist", "always sees upside and opportunity",
                      lambda q: "worth it - GenAI skills pay back fast in a growing market")
skeptic  = Specialist("Skeptic",  "questions assumptions and finds the risks",
                      lambda q: "risky - only pays off if you finish it and actually apply it")

q = "Is the 14999 GenAI course worth it?"
for agent in (optimist, skeptic):
    print(" ", agent.answer(q))
print("same question, different answers - the BACKSTORY changed how each agent reasoned")

# Output:
#   [Optimist] worth it - GenAI skills pay back fast in a growing market
#   [Skeptic] risky - only pays off if you finish it and actually apply it
# same question, different answers - the BACKSTORY changed how each agent reasoned

- Each `Specialist` carries a `backstory` that stands in for a system prompt.
- The Optimist and the Skeptic get the same question but return opposite answers.
- The only difference is the backstory - it changed how each agent reasoned.
- In a real framework this backstory is sent on every call, so it shapes output AND costs tokens (Step 1).

#### 💡 What just happened

⚡ What just happened? A specialist is role + goal + backstory + tools; the backstory is a system prompt that shapes reasoning. Tradeoff: specialization improves focus and tool selection, but each role prompt is paid on every call. Design the smallest set of roles that covers the work.

- Edit an agent's backstory and watch the same question get a different answer
- Toggle the role on and off to see the shift

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Coordination Topologies

### Coordination Topologies

Sequential, supervisor/route, broadcast, debate. Same specialists, four ways to wire them together.

Given specialists, how do they collaborate? There are four canonical **topologies**, and the whole art of multi-agent design is picking the right one. **Sequential** is an assembly line: each agent builds on the previous one’s output (research → analyze → write). **Supervisor / route** is a dispatcher: a leader reads the task and sends it to the *one* best agent - cheapest, because only one runs. **Broadcast** fans the same task out to *all* agents in parallel, then synthesizes their answers - good for independent perspectives. **Debate** has agents challenge each other over rounds, then a judge decides - slower, but it surfaces disagreement a single agent would miss. Below, one cast runs through all four, so you can see how the same team produces different behavior depending only on the wiring.

> 🌐 **Analogy**
>
> These are **org charts**. Sequential is an assembly line. Supervisor/route is a **switchboard operator** who connects your call to the right department. Broadcast is a **brainstorm** where everyone answers and a chair sums up. Debate is a **panel discussion** with a moderator. Same people, very different meetings.

You need the cheapest topology: only the single most relevant agent should run. Which one?

**📝 `03_topologies.py`** - *runnable*

In [ ]:
# FOUR WAYS A TEAM COORDINATES. Same cast, same goal, four SHAPES. The think step is scripted.
KB = {"genai": "GenAI: 14999 INR, 146 hrs, high demand"}
def researcher(ctx): return "facts: " + KB["genai"]
def analyst(ctx):    return "analysis: strong ROI if applied  <- " + ctx
def writer(ctx):     return "report: enrol; strong ROI if applied"

def sequential(agents):                 # ASSEMBLY LINE: each agent builds on the previous output
    ctx = ""
    for a in agents: ctx = a(ctx)
    return ctx

def route(agents, leader_pick):         # SUPERVISOR/ROUTE: a leader sends the task to ONE best agent
    return agents[leader_pick]("")

def broadcast(agents):                  # BROADCAST: fan out to ALL in parallel, then synthesise
    views = [a(" (parallel)") for a in agents]
    return f"synthesis of {len(views)} views -> enrol (consensus)"

def debate(pro, con):                   # DEBATE: agents critique each other, then a judge decides
    return f"judge: weighed [{pro('')}] vs [{con('')}] -> enrol, but finish it"

team = [researcher, analyst, writer]
print("sequential:", sequential(team))
print("route     :", route(team, 0), "  (only the researcher ran - cheapest)")
print("broadcast :", broadcast([researcher, analyst]))
print("debate    :", debate(lambda c: "pro: high upside", lambda c: "con: only if applied"))

# Output:
# sequential: report: enrol; strong ROI if applied
# route     : facts: GenAI: 14999 INR, 146 hrs, high demand   (only the researcher ran - cheapest)
# broadcast : synthesis of 2 views -> enrol (consensus)
# debate    : judge: weighed [pro: high upside] vs [con: only if applied] -> enrol, but finish it

- **Sequential** threads context through the team: research → analyze → write.
- **Route** runs only the researcher - one agent, the cheapest topology.
- **Broadcast** runs agents in parallel and synthesizes their views.
- **Debate** pits a pro and a con, then a judge weighs both - the same cast, four shapes.

#### 💡 What just happened

⚡ What just happened? The four topologies are sequential, supervisor/route, broadcast, and debate - the same specialists wired four ways. Tradeoff: route is cheapest (one agent), broadcast and debate cost the most but surface independent views. Match the topology to the task, not the fashion. CrewAI and Agno (next) are these shapes in a framework.

- Flip among sequential / route / broadcast / debate
- Watch messages flow through the team differently for each

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: CrewAI: Role-Based Crews

### CrewAI: Role-Based Crews

Define agents with roles + tasks, group them in a Crew, and run sequential or hierarchical. The role abstraction, made easy.

**CrewAI** makes the role-based team its core abstraction. You define **Agents** (role / goal / backstory / tools), write **Tasks** (a description + expected output, assigned to an agent, optionally depending on other tasks via `context`), and group them in a **Crew** with a **process**: `Process.sequential` runs the tasks in order (each task’s output feeds the next), while `Process.hierarchical` adds a manager (you must supply a `manager_llm`) that *delegates* tasks instead of following a fixed line. `crew.kickoff()` runs it. For production, CrewAI’s **Flows** (event-driven `@start`/`@listen`/`@router` with typed state) wrap crews and replaced the deprecated Pipelines. The block below is the exact shape of Step 3’s sequential topology - now in a framework.

> 🎬 **Analogy**
>
> CrewAI is a **film crew**. Each person has a role (director, camera, editor), there is a call sheet of tasks in order, and the shoot runs down the list. Switch to **hierarchical** and you add a producer who reassigns work on the fly instead of following the fixed schedule.

A CrewAI task has `context=[research_task]`. What does that do?

**📝 `04_crewai.py`** - *illustrative*

In [ ]:
# CREWAI - role-based crews. Each agent has a role/goal/BACKSTORY; a Crew runs the tasks in order.
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

@tool
def search_courses(topic: str) -> str:
    "Search the Netsetos course catalog by topic."
    return {"genai": "GenAI: 14999 INR, 146 hrs, high demand"}.get(topic.lower(), "not found")

researcher = Agent(role="Course Researcher", goal="Find the course + demand facts",
                   backstory="An edtech researcher who digs up the numbers.",   # backstory shapes reasoning
                   tools=[search_courses], llm="anthropic/claude-sonnet-4-6")
writer = Agent(role="Advisor", goal="Write a clear recommendation",
               backstory="A student-friendly writer who turns facts into advice.",
               llm="anthropic/claude-sonnet-4-6")

research = Task(description="Research the GenAI course and its demand.",
                expected_output="Key facts + demand.", agent=researcher)
advise = Task(description="Write a short recommendation from the research.",
              expected_output="A 3-line advisory.", agent=writer, context=[research])

crew = Crew(agents=[researcher, writer], tasks=[research, advise], process=Process.sequential)
result = crew.kickoff()          # sequential: the researcher's output feeds the writer's task
print(result)

# Output: (illustrative minimal example - needs `pip install crewai crewai-tools` + a key.) The Crew runs
#   the tasks in order; context=[research] hands the researcher's output to the writer. Swap to
#   process=Process.hierarchical (with a manager_llm) to let a manager delegate instead of a fixed line.

- `Agent` = role / goal / **backstory** / tools - the specialist from Step 2, in CrewAI.
- `Task` assigns work to an agent; `context=[research]` hands one task’s output to the next.
- `Crew(process=Process.sequential)` runs the tasks in order; `kickoff()` starts it.
- Swap to `Process.hierarchical` (with a `manager_llm`) for a manager that delegates; Flows wrap crews for production.

#### 💡 What just happened

⚡ What just happened? CrewAI packages the role-based team: Agent (role/goal/backstory/tools) + Task + Crew(process). Tradeoff: the role abstraction is fast to build and read, but every role prompt is injected each call (the Step 1 tax). Use sequential for a fixed pipeline, hierarchical when a manager should delegate.

- Build a crew of roles and tasks; run it
- Toggle sequential vs hierarchical and watch the crew execute

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Agno: Lightweight Teams & the Four Modes

### Agno: Lightweight Teams & the Four Modes

Agent + Team(members, mode). One mode switch changes the whole coordination shape. Model-agnostic and fast.

**Agno** takes a lighter, mode-first approach. You build **Agents** (name / role / model / tools - tools are passed directly), then a **Team** with `members=[...]` and a **`mode`** that *is* the coordination topology: **`coordinate`** (a leader delegates and synthesizes), **`route`** (dispatch to the single best member), **`broadcast`** (send to all members in parallel, collect responses), and **`tasks`** (decompose into discrete units). Switching topology is a one-word change - the same mental map as Step 3. Agno is **model-agnostic** (swap the model in one line) and fast to instantiate, and its AgentOS runtime serves teams behind a FastAPI backend. (Note: the mode formerly called “collaborate” is now `broadcast`.)

> ☎️ **Analogy**
>
> Agno’s Team is a **help desk with a switchboard**. In `route` mode the operator connects you to the one right specialist; in `broadcast` mode they conference everyone in at once; in `coordinate` mode a lead takes your request, parcels it out, and hands you back one answer. One dial changes how the whole desk works.

In Agno you want the leader to send each query to only the single best agent. Which mode?

**📝 `05_agno.py`** - *illustrative*

In [ ]:
# AGNO - lightweight teams. A Team has `members` and a `mode`; here `route` sends each query to ONE agent.
from agno.agent import Agent
from agno.team import Team
from agno.models.anthropic import Claude

def search_courses(topic: str) -> str:
    "Search the Netsetos course catalog by topic."
    return {"genai": "GenAI: 14999 INR, 146 hrs"}.get(topic.lower(), "not found")

researcher = Agent(name="Researcher", role="Find course facts",
                   model=Claude(id="claude-sonnet-4-6"), tools=[search_courses])   # tools passed directly
advisor = Agent(name="Advisor", role="Recommend a course to the student",
                model=Claude(id="claude-sonnet-4-6"))

team = Team(name="Advisory", members=[researcher, advisor],   # members=, not agents=
            mode="route",                                     # coordinate | route | broadcast | tasks
            model=Claude(id="claude-sonnet-4-6"))             # the team leader
team.print_response("What does the GenAI course cost?")

# Output: (illustrative minimal example - needs `pip install agno anthropic` + a key.) mode="route" makes the
#   leader dispatch to the single best member (the Researcher). Other modes: coordinate (delegate + synthesise),
#   broadcast (all members in parallel), tasks (decompose into units). Agno is model-agnostic - swap Claude for any.

- `Team(members=[...], mode=...)` - `members=` (not `agents=`), and `mode` is the topology.
- The four modes are `coordinate` / `route` / `broadcast` / `tasks` - Step 3’s shapes as one switch.
- Tools are passed directly to the agent; Agno is model-agnostic, so swapping the model is one line.
- `broadcast` is the current name for what older docs called “collaborate.”

#### 💡 What just happened

⚡ What just happened? Agno makes the topology a one-word `mode`: coordinate / route / broadcast / tasks, over a Team of `members`. Tradeoff / vs CrewAI: Agno is lighter and model-agnostic with modes as first-class; CrewAI leads with the role/crew abstraction. Same patterns, two mental models - pick the one that fits your team.

- Pick a team mode: coordinate / route / broadcast / tasks
- Watch the leader route the query to member(s) accordingly

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Handoffs & the Aggregation Problem

### Handoffs & the Aggregation Problem

How work passes between agents - and why parallel agents on coupled work make conflicting choices someone must reconcile.

Agents collaborate by **handing off** work, and how they hand off matters. Pass the **full history** (as OpenAI’s SDK transfers do) and the next agent has all the context but the token bill grows; spawn a **clean-slate** sub-agent (as Claude’s sub-agents do) and you keep contexts isolated but must brief it carefully. And here is the trap that sinks naive multi-agent systems - the **aggregation problem**: when agents work in **parallel on coupled work**, each sees only part of the picture and makes choices that *clash* (two coders name the same button differently), so a later step has to reconcile them. This is exactly why Cognition argues you should *not* parallelize coupled tasks like coding. The fix is to keep coupled work sequential (or give one supervisor the full context to decide), and reserve parallelism for genuinely independent subtasks.

> ✍️ **Analogy**
>
> It is **two co-authors writing different chapters who never compare notes**. One calls the hero ‘Sam,’ the other ‘Sammy’; one kills a character the other keeps alive. The book does not add up until an **editor** reconciles them - the coordination cost a single author never pays.

Two agents build the same checkout button in parallel, each seeing half the spec. Likely result?

**📝 `06_handoffs.py`** - *runnable*

In [ ]:
# HANDOFFS + THE AGGREGATION PROBLEM. Two coders work IN PARALLEL on partial context and pick
# clashing choices; a later step has to reconcile them (Cognition: do not parallelise coupled work).
def coder_a(spec): return {"button": "Buy Now",  "color": "green"}   # saw only half the spec
def coder_b(spec): return {"button": "Purchase", "color": "blue"}    # saw the other half

a, b = coder_a("checkout"), coder_b("checkout")
conflict = {k: (a[k], b[k]) for k in a if a[k] != b[k]}
print("agent A:", a)
print("agent B:", b)
print("CONFLICT (parallel agents on partial context):", conflict)

# Fix: a SUPERVISOR with the full context decides once, consistently.
resolved = {"button": "Buy Now", "color": "green"}
print("resolved by a supervisor / shared context:", resolved)
print("lesson: parallel agents on COUPLED work make conflicting choices; keep coupled work sequential")

# Output:
# agent A: {'button': 'Buy Now', 'color': 'green'}
# agent B: {'button': 'Purchase', 'color': 'blue'}
# CONFLICT (parallel agents on partial context): {'button': ('Buy Now', 'Purchase'), 'color': ('green', 'blue')}
# resolved by a supervisor / shared context: {'button': 'Buy Now', 'color': 'green'}
# lesson: parallel agents on COUPLED work make conflicting choices; keep coupled work sequential

- Two coders, each on *partial* context, pick a different button label and color.
- The `conflict` dict shows exactly where their parallel choices clash.
- A supervisor with the *full* context resolves it to one consistent answer.
- Lesson: keep coupled work sequential (or under one supervisor); parallelize only independent subtasks.

#### 💡 What just happened

⚡ What just happened? Handoffs pass work (full-history vs clean-slate), and parallel agents on coupled work hit the aggregation problem - conflicting choices someone must reconcile. Tradeoff: full-history is context-rich but expensive; clean-slate is cheap but needs careful briefing. Keep coupled work sequential; parallelize independent work only.

- Run two agents in parallel on partial context and watch them conflict
- A supervisor with full context reconciles the clash

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: When Multi-Agent Wins (and When It Hurts)

### When Multi-Agent Wins (and When It Hurts)

Parallel, isolation, and critic tasks win; coupled and sequential tasks hurt. Plus: MCP and A2A make the framework choice less permanent.

Now the judgment, grounded in what practitioners actually found. Multi-agent **wins** when: the work is **parallelizable and independent** (Anthropic’s breadth-first research - exploring one source does not depend on another); when **context isolation** helps (a single agent’s context gets polluted, separate clean contexts do better); and when you want a **critic/reviewer** (a second agent catches the first’s mistakes). It **hurts** when the work is **coupled and sequential** - like writing code, where every decision constrains the next, so parallel agents conflict (Cognition), and where Google found adding agents actively degrades performance. When you do reach for a team, note that **MCP** (agent↔tools) and **A2A** (agent↔agent) make tools and agents interoperable across frameworks - so the CrewAI-vs-Agno choice is less permanent than it looks. Start from the pattern and the economics; pick the framework last.

> 🤝 **Analogy**
>
> It is **solo versus a team hire**. Bring in a team when the work genuinely splits into independent parts, or you want a second pair of eyes to review. For one coherent, tightly-coupled job, a great solo hire is faster, cheaper, and never contradicts themselves. The mistake is hiring a team for a solo job.

You are writing one feature end-to-end where each step depends on the last. Single or multi-agent?

**📝 `07_when_multi_agent.py`** - *runnable*

In [ ]:
# WHEN TO GO MULTI-AGENT - default to a single agent; escalate only when the task earns it.
def choose(sequential_pipeline, parallelizable, needs_isolation, needs_critic):
    if sequential_pipeline: return "single agent", "each step constrains the next; parallel agents would conflict (Cognition)"
    if parallelizable:      return "multi-agent",  "independent subtasks run in parallel (breadth-first research)"
    if needs_isolation:     return "multi-agent",  "isolate contexts so one subtask does not pollute another"
    if needs_critic:        return "multi-agent",  "a separate reviewer catches the worker's mistakes"
    return "single agent", "one well-designed agent is cheaper and usually enough"

cases = [
    ("write a feature end to end", dict(sequential_pipeline=1, parallelizable=0, needs_isolation=0, needs_critic=0)),
    ("research 8 sources at once", dict(sequential_pipeline=0, parallelizable=1, needs_isolation=0, needs_critic=0)),
    ("draft then review code",     dict(sequential_pipeline=0, parallelizable=0, needs_isolation=0, needs_critic=1)),
    ("answer one simple question", dict(sequential_pipeline=0, parallelizable=0, needs_isolation=0, needs_critic=0)),
]
for name, flags in cases:
    pick, why = choose(**flags)
    print(f"  {name:28s} -> {pick:12s} ({why})")

# Output:
#   write a feature end to end   -> single agent (each step constrains the next; parallel agents would conflict (Cognition))
#   research 8 sources at once   -> multi-agent  (independent subtasks run in parallel (breadth-first research))
#   draft then review code       -> multi-agent  (a separate reviewer catches the worker's mistakes)
#   answer one simple question   -> single agent (one well-designed agent is cheaper and usually enough)

- A coupled, sequential feature → **single agent** (parallel agents would conflict).
- Independent, parallelizable research → **multi-agent** (the case where teams win).
- Draft-then-review → **multi-agent** (the critic/reviewer pattern).
- A simple question → **single agent** - the default, and usually enough.

#### 💡 What just happened

⚡ What just happened? Multi-agent wins on parallel / isolation / critic tasks and hurts on coupled / sequential ones; MCP + A2A make the framework choice swappable. Tradeoff / the whole lesson: a team buys you specialization and parallelism at three-to-ten times the tokens, so reach for it only when the task earns the tax.

- Set the task properties: parallel? coupled? needs a critic?
- Watch it route to single vs multi with the reason

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> A team of specialists can beat one agent - but it costs roughly three to ten times the tokens, and with an equal budget one agent often matches it (Step 1). A specialist is role + goal + **backstory** (a system prompt that shapes reasoning) + tools (Step 2). Those specialists are wired with one of four **topologies** - sequential, supervisor/route, broadcast, debate (Step 3). **CrewAI** packages this as Agents + Tasks + a Crew with a sequential or hierarchical process (Step 4); **Agno** packages it as a Team of `members` with a one-word `mode` - coordinate / route / broadcast / tasks (Step 5). Agents **hand off** work, and parallel agents on coupled work hit the **aggregation problem** (Step 6). And the judgment: multi-agent wins on parallel / isolation / critic tasks and hurts on coupled / sequential ones (Step 7). Reach for a team only when the task earns its tax.

| Topology | Shape | Cost | Best for |
|---|---|---|---|
| Sequential | assembly line (A → B → C) | medium | a fixed, ordered pipeline |
| Supervisor / route | leader dispatches to one | lowest (one agent) | known categories, specialist dispatch |
| Broadcast | all in parallel, then synthesize | high | independent perspectives / breadth |
| Debate | agents critique, a judge decides | high | surfacing disagreement / hard calls |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now design, build, and cost a multi-agent team. Next: **computer-use agents** in Lesson 11.6; **human-in-the-loop** approval gates for agent teams in Lesson 11.10; and **agent evaluation & security** - how to catch multi-agent failures - in Lesson 11.11. The references are Anthropic’s [multi-agent research system](https://www.anthropic.com/engineering/multi-agent-research-system), Cognition’s essay *Don’t Build Multi-Agents*, and the [CrewAI](https://github.com/crewaiinc/crewai) and [Agno](https://docs.agno.com/teams/overview) docs.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Multi-AgentSystems: CrewAI & Agno**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-11_5.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-11.5.html` - regenerate this notebook whenever the source HTML is updated.*
